# Module 2: Two-Way Fixed Effects (TWFE) Practice

## Learning Objectives

By completing this notebook, you will:
1. Understand when to include time fixed effects
2. Implement two-way fixed effects (entity + time)
3. Interpret TWFE coefficients correctly
4. Apply TWFE to real economic policy questions
5. Recognize limitations of TWFE with treatment heterogeneity

## Prerequisites

- Module 2.1: FE Implementation (completed)
- Module 2.2: FE Diagnostics (completed)
- Understanding of omitted variable bias

## Estimated Time

75-90 minutes

---

## Setup

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from linearmodels.panel import PanelOLS

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Set random seed
np.random.seed(42)

print("✅ Setup complete")

In [ ]:
section_divider("1. The Two-Way Fixed Effects Model")

In [ ]:
learning_objectives(["Understand when to include time fixed effects", "Implement two-way fixed effects (entity + time)", "Interpret TWFE coefficients correctly", "Apply TWFE to real economic policy questions", "Recognize limitations of TWFE with treatment heterogeneity"])

In [ ]:
apply_course_theme()
apply_plot_theme()

## 1. The Two-Way Fixed Effects Model

### Model Specification

$$y_{it} = \alpha_i + \lambda_t + X_{it}\beta + \epsilon_{it}$$

where:
- $\alpha_i$ = **entity fixed effect** (controls for time-invariant entity characteristics)
- $\lambda_t$ = **time fixed effect** (controls for common shocks affecting all entities)
- $X_{it}$ = time-varying predictors
- $\beta$ = causal effect of X on y

### When to Include Time Fixed Effects?

**Include $\lambda_t$ when:**
1. Common shocks affect all entities (economic recessions, policy changes)
2. Aggregate trends may confound relationship (technological progress)
3. Seasonality affects outcome (quarterly/monthly data)

### What Do Time FE Control For?

- Economic cycles (recessions, booms)
- Inflation/price level changes
- Federal policy changes
- Technological trends
- Measurement changes over time

In [ ]:
section_divider("2. Example: Minimum Wage and Employment")

## 2. Example: Minimum Wage and Employment

### Research Question

**Does increasing the minimum wage reduce employment?**

This is a classic question in labor economics. We'll simulate state-level panel data.

### Data Generating Process

- States differ in baseline employment (entity FE)
- National recessions affect all states (time FE)
- Minimum wage has causal effect on employment

In [ ]:
# Generate state-level panel data
n_states = 50
n_years = 15
years = np.arange(2005, 2005 + n_years)

# State fixed effects (baseline employment rate)
state_effects = np.random.randn(n_states) * 2 + 60  # Mean employment rate 60%

# Time fixed effects (business cycle)
time_effects = np.zeros(n_years)
time_effects[0:3] = 0  # Normal times
time_effects[3:6] = -3  # Recession 2008-2010
time_effects[6:10] = 1  # Recovery
time_effects[10:] = 2  # Boom

# Create panel
data = []
for state in range(n_states):
    # State-specific minimum wage policy
    # Some states raise min wage more than others
    state_policy_aggressiveness = np.random.uniform(0.5, 1.5)
    
    for t, year in enumerate(years):
        # Minimum wage increases over time (more in some states)
        min_wage = 5.5 + 0.3 * t + state_policy_aggressiveness * 0.2 * t + np.random.randn() * 0.3
        
        # Employment rate (true causal effect: beta = -0.5)
        employment = (state_effects[state] + 
                      time_effects[t] + 
                      -0.5 * min_wage +  # True causal effect
                      np.random.randn() * 1.5)
        
        data.append({
            'state_id': state + 1,
            'year': year,
            'employment_rate': employment,
            'min_wage': min_wage
        })

df_minwage = pd.DataFrame(data)

print("Minimum Wage Panel Data Generated")
print(f"  States: {n_states}")
print(f"  Years: {n_years} (2005-2019)")
print(f"  True causal effect: -0.5")
print(f"  (1 dollar min wage increase → 0.5 point decrease in employment)")
print(f"\nFirst 20 rows:")
print(df_minwage.head(20))

### Visualize the Data

In [ ]:
# Average employment rate over time
time_avg = df_minwage.groupby('year')[['employment_rate', 'min_wage']].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Employment rate over time
axes[0].plot(time_avg.index, time_avg['employment_rate'], 
             marker='o', linewidth=2, markersize=8)
axes[0].axvspan(2008, 2010, alpha=0.2, color='red', label='Recession')
axes[0].set_xlabel('Year', fontsize=11)
axes[0].set_ylabel('Employment Rate (%)', fontsize=11)
axes[0].set_title('Average Employment Rate Over Time', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Panel 2: Minimum wage over time
axes[1].plot(time_avg.index, time_avg['min_wage'], 
             marker='o', linewidth=2, markersize=8, color='green')
axes[1].set_xlabel('Year', fontsize=11)
axes[1].set_ylabel('Minimum Wage ($)', fontsize=11)
axes[1].set_title('Average Minimum Wage Over Time', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Note: Employment drops during recession (2008-2010)")
print("      This time effect must be controlled for!")

In [ ]:
section_divider("3. Estimating with Different Fixed Effects")

## 3. Estimating with Different Fixed Effects

Let's compare:
1. **Pooled OLS** (no fixed effects)
2. **Entity FE only** (state fixed effects)
3. **Two-way FE** (state + time fixed effects)

In [ ]:
# Prepare data
df_panel = df_minwage.set_index(['state_id', 'year'])

# Model 1: Pooled OLS
X_pooled = sm.add_constant(df_minwage['min_wage'])
y_pooled = df_minwage['employment_rate']
model_pooled = sm.OLS(y_pooled, X_pooled).fit()

# Model 2: Entity FE only
model_fe = PanelOLS(
    dependent=df_panel['employment_rate'],
    exog=df_panel[['min_wage']],
    entity_effects=True
)
results_fe = model_fe.fit(cov_type='clustered', cluster_entity=True)

# Model 3: Two-way FE
model_twfe = PanelOLS(
    dependent=df_panel['employment_rate'],
    exog=df_panel[['min_wage']],
    entity_effects=True,
    time_effects=True  # Add time fixed effects
)
results_twfe = model_twfe.fit(cov_type='clustered', cluster_entity=True)

# Compare results
print("Comparison: Different Fixed Effects Specifications")
print("=" * 70)
print(f"{'Model':<25} {'Beta':>12} {'Std Err':>12} {'t-stat':>10}")
print("=" * 70)

beta_pooled = model_pooled.params['min_wage']
se_pooled = model_pooled.bse['min_wage']
t_pooled = model_pooled.tvalues['min_wage']

beta_fe = results_fe.params['min_wage']
se_fe = results_fe.std_errors['min_wage']
t_fe = results_fe.tstats['min_wage']

beta_twfe = results_twfe.params['min_wage']
se_twfe = results_twfe.std_errors['min_wage']
t_twfe = results_twfe.tstats['min_wage']

print(f"{'True effect':<25} {-0.5:>12.4f} {'':>12} {'':>10}")
print(f"{'Pooled OLS':<25} {beta_pooled:>12.4f} {se_pooled:>12.4f} {t_pooled:>10.2f}")
print(f"{'Entity FE only':<25} {beta_fe:>12.4f} {se_fe:>12.4f} {t_fe:>10.2f}")
print(f"{'Two-way FE (TWFE)':<25} {beta_twfe:>12.4f} {se_twfe:>12.4f} {t_twfe:>10.2f}")

print("\n" + "=" * 70)
print("Bias from true effect (-0.5):")
print(f"  Pooled OLS:      {abs(beta_pooled - (-0.5)):.4f}")
print(f"  Entity FE only:  {abs(beta_fe - (-0.5)):.4f}")
print(f"  Two-way FE:      {abs(beta_twfe - (-0.5)):.4f}")

print("\n✅ TWFE is closest to true effect!")
print("   Time effects control for recession and aggregate trends.")

### Extract and Visualize Time Fixed Effects

In [ ]:
# Get estimated time effects from TWFE
# Note: linearmodels doesn't directly expose time FE, but we can recover them

# Manually compute time effects by demeaning
# First demean by entity
entity_means = df_minwage.groupby('state_id')[['employment_rate', 'min_wage']].transform('mean')
df_demeaned = df_minwage.copy()
df_demeaned['emp_demeaned'] = df_minwage['employment_rate'] - entity_means['employment_rate']
df_demeaned['wage_demeaned'] = df_minwage['min_wage'] - entity_means['min_wage']

# Residuals after removing entity FE and X*beta
df_demeaned['resid_after_fe'] = (df_demeaned['emp_demeaned'] - 
                                  beta_twfe * df_demeaned['wage_demeaned'])

# Time effects are year averages of these residuals
time_fe_estimated = df_demeaned.groupby('year')['resid_after_fe'].mean()

# Plot true vs estimated time effects
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(years, time_effects, marker='o', linewidth=2, markersize=8, 
        label='True Time Effects', color='blue')
ax.plot(time_fe_estimated.index, time_fe_estimated.values, 
        marker='s', linewidth=2, markersize=8, 
        label='Estimated Time Effects', color='red', linestyle='--')

ax.axhline(y=0, color='gray', linestyle='-', linewidth=1, alpha=0.5)
ax.axvspan(2008, 2010, alpha=0.2, color='red', label='Recession')

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Time Fixed Effect', fontsize=12)
ax.set_title('Time Fixed Effects: True vs Estimated', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Time fixed effects capture:")
print("  - Recession impact (2008-2010): negative shock")
print("  - Recovery and boom: positive shocks")
print("  - These are common to all states")

In [ ]:
section_divider("4. Interpretation of TWFE Coefficients")

## 4. Interpretation of TWFE Coefficients

### What Does the TWFE Coefficient Mean?

The TWFE coefficient $\beta$ represents:

> "The expected change in Y when X increases by 1, **within the same entity over time**, after removing common time trends shared by all entities."

### Example Interpretation

For our minimum wage analysis:

*"A 1 dollar increase in the minimum wage within a state is associated with a {beta} percentage point change in employment, controlling for state-specific characteristics and national economic conditions."*

In [ ]:
# Formal interpretation
print("TWFE Estimation Results Summary")
print("=" * 70)
print(f"Dependent variable: Employment Rate (%)")
print(f"Independent variable: Minimum Wage ($)")
print(f"")
print(f"Coefficient: {beta_twfe:.4f}")
print(f"Std Error: {se_twfe:.4f} (clustered by state)")
print(f"95% CI: [{beta_twfe - 1.96*se_twfe:.4f}, {beta_twfe + 1.96*se_twfe:.4f}]")
print(f"t-statistic: {t_twfe:.3f}")
print(f"")
print(f"Interpretation:")
print(f"  A $1 increase in minimum wage is associated with a")
print(f"  {beta_twfe:.3f} percentage point change in employment rate,")
print(f"  controlling for:")
print(f"    - Time-invariant state characteristics (entity FE)")
print(f"    - Common economic shocks and trends (time FE)")

if abs(t_twfe) > 1.96:
    print(f"\n  ✅ Effect is statistically significant at 5% level")
else:
    print(f"\n  ⚠️  Effect is not statistically significant")

In [ ]:
section_divider("5. Testing Need for Time Fixed Effects")

## 5. Testing Need for Time Fixed Effects

### F-Test: Are Time Fixed Effects Jointly Significant?

$$H_0: \lambda_1 = \lambda_2 = ... = \lambda_T = 0$$

If we reject, time FE are important.

In [ ]:
def f_test_time_effects(df, entity_col, time_col, y_col, X_cols):
    """
    F-test for significance of time fixed effects.
    
    Compares entity FE vs two-way FE models.
    
    Parameters
    ----------
    df : pd.DataFrame
    entity_col : str
    time_col : str
    y_col : str
    X_cols : list
    
    Returns
    -------
    dict
        F-statistic and p-value
    """
    df_panel = df.set_index([entity_col, time_col])
    
    # Model 1: Entity FE only
    model1 = PanelOLS(
        dependent=df_panel[y_col],
        exog=df_panel[X_cols],
        entity_effects=True
    )
    results1 = model1.fit()
    ssr_restricted = results1.resid_ss
    
    # Model 2: Two-way FE
    model2 = PanelOLS(
        dependent=df_panel[y_col],
        exog=df_panel[X_cols],
        entity_effects=True,
        time_effects=True
    )
    results2 = model2.fit()
    ssr_full = results2.resid_ss
    
    # F-test
    n_time = df[time_col].nunique()
    q = n_time - 1  # Number of time FE
    n = len(df)
    k = len(X_cols)
    
    f_stat = ((ssr_restricted - ssr_full) / q) / (ssr_full / (n - k - q))
    p_value = 1 - stats.f.cdf(f_stat, q, n - k - q)
    
    return {
        'f_statistic': f_stat,
        'df1': q,
        'df2': n - k - q,
        'p_value': p_value
    }

# Test for time effects
f_test_result = f_test_time_effects(
    df_minwage, 'state_id', 'year', 'employment_rate', ['min_wage']
)

print("F-Test for Time Fixed Effects")
print("=" * 70)
print(f"H0: All time fixed effects are zero")
print(f"")
print(f"F-statistic: {f_test_result['f_statistic']:.2f}")
print(f"Degrees of freedom: ({f_test_result['df1']}, {f_test_result['df2']})")
print(f"P-value: {f_test_result['p_value']:.4e}")

if f_test_result['p_value'] < 0.01:
    print("\n✅ Reject H0: Time fixed effects are jointly significant!")
    print("   Conclusion: Include time FE in the model.")
else:
    print("\n⚠️  Cannot reject H0: Time FE may not be necessary.")

## 6. Practical Application: Diff-in-Diff as TWFE

**Difference-in-differences** (DiD) is a special case of TWFE.

### Classic DiD Setup

- **Treatment group:** Some states raise minimum wage
- **Control group:** Other states don't
- **Pre/Post periods:** Before and after policy change

### DiD as TWFE

$$y_{it} = \alpha_i + \lambda_t + \delta \cdot (Treated_i \times Post_t) + \epsilon_{it}$$

The treatment effect $\delta$ can be estimated with TWFE.

In [ ]:
# Create DiD scenario
# Treatment: States 1-25 raise minimum wage in 2012
df_did = df_minwage.copy()

# Treatment indicator
df_did['treated'] = (df_did['state_id'] <= 25).astype(int)
df_did['post'] = (df_did['year'] >= 2012).astype(int)
df_did['treated_post'] = df_did['treated'] * df_did['post']

# Add treatment effect (raise min wage by $2 in treated states after 2012)
df_did.loc[df_did['treated_post'] == 1, 'min_wage'] += 2

# True treatment effect on employment: -0.5 * 2 = -1.0

# Estimate TWFE with treatment indicator
df_did_panel = df_did.set_index(['state_id', 'year'])

model_did = PanelOLS(
    dependent=df_did_panel['employment_rate'],
    exog=df_did_panel[['treated_post']],  # Treatment effect
    entity_effects=True,
    time_effects=True
)
results_did = model_did.fit(cov_type='clustered', cluster_entity=True)

print("Difference-in-Differences via TWFE")
print("=" * 70)
print("Treatment: States 1-25 raise minimum wage by $2 in 2012")
print("Control: States 26-50 (no policy change)")
print(f"")
print("Treatment Effect Estimate:")
print(f"  Coefficient: {results_did.params['treated_post']:.4f}")
print(f"  Std Error: {results_did.std_errors['treated_post']:.4f}")
print(f"  t-statistic: {results_did.tstats['treated_post']:.3f}")
print(f"  p-value: {results_did.pvalues['treated_post']:.4e}")
print(f"")
print(f"Interpretation:")
print(f"  Raising minimum wage by $2 led to a {results_did.params['treated_post']:.3f}")
print(f"  percentage point change in employment.")
print(f"")
print(f"Expected effect (true): -1.0")
print(f"Estimated effect: {results_did.params['treated_post']:.4f}")

if abs(results_did.tstats['treated_post']) > 1.96:
    print(f"\n✅ Effect is statistically significant")
else:
    print(f"\n⚠️  Effect is not statistically significant")

### Visualize Parallel Trends

In [ ]:
# Average employment by treatment status and year
trends = df_did.groupby(['year', 'treated'])['employment_rate'].mean().unstack()

fig, ax = plt.subplots(figsize=(12, 6))

# Plot treated and control groups
ax.plot(trends.index, trends[0], marker='o', linewidth=2, markersize=8,
        label='Control States', color='blue')
ax.plot(trends.index, trends[1], marker='s', linewidth=2, markersize=8,
        label='Treated States', color='red')

# Mark treatment time
ax.axvline(x=2012, color='black', linestyle='--', linewidth=2, 
           label='Treatment (2012)')

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Employment Rate (%)', fontsize=12)
ax.set_title('Parallel Trends: Employment by Treatment Status', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key assumption: Parallel trends in pre-treatment period")
print("  Before 2012, treated and control should have similar trends")
print("  After 2012, gap = treatment effect")

---

## Exercises

### Exercise 1: Event Study Design

**Task:** Estimate the treatment effect **separately for each year** after treatment (event study).

Create indicators:
- `year_-3`, `year_-2`, `year_-1` (pre-treatment)
- `year_0`, `year_+1`, `year_+2`, ... (post-treatment)

Omit one pre-treatment year as base category.

**Hints:**
- Create relative time variable: `years_since_treatment`
- Generate dummies for each relative year
- Plot coefficients with confidence intervals

In [ ]:
# YOUR CODE HERE
def event_study_twfe(df, entity_col, time_col, y_col, 
                     treatment_col, treatment_year):
    """
    Estimate event study coefficients.
    
    Parameters
    ----------
    df : pd.DataFrame
    entity_col : str
    time_col : str
    y_col : str
    treatment_col : str
        Binary indicator for treatment group
    treatment_year : int
        Year treatment begins
    
    Returns
    -------
    pd.DataFrame
        Coefficients and SEs by relative time
    """
    # TODO: Implement event study
    pass

In [ ]:
# SOLUTION (hidden)
def event_study_twfe_solution(df, entity_col, time_col, y_col, 
                               treatment_col, treatment_year):
    """Estimate event study coefficients."""
    df_es = df.copy()
    
    # Create relative time
    df_es['rel_time'] = df_es[time_col] - treatment_year
    
    # Create event time dummies (interacted with treatment)
    # Omit t=-1 as base period
    rel_times = sorted(df_es['rel_time'].unique())
    rel_times = [t for t in rel_times if t != -1]  # Omit -1
    
    for t in rel_times:
        col_name = f'event_t{t:+d}'
        df_es[col_name] = ((df_es['rel_time'] == t) & 
                           (df_es[treatment_col] == 1)).astype(int)
    
    # Get column names for regression
    event_cols = [f'event_t{t:+d}' for t in rel_times]
    
    # Estimate TWFE with event dummies
    df_es_panel = df_es.set_index([entity_col, time_col])
    
    model = PanelOLS(
        dependent=df_es_panel[y_col],
        exog=df_es_panel[event_cols],
        entity_effects=True,
        time_effects=True
    )
    results = model.fit(cov_type='clustered', cluster_entity=True)
    
    # Extract coefficients
    event_study_results = pd.DataFrame({
        'rel_time': rel_times,
        'coef': [results.params[f'event_t{t:+d}'] for t in rel_times],
        'se': [results.std_errors[f'event_t{t:+d}'] for t in rel_times]
    })
    
    # Add base period (t=-1) with coef=0
    base_row = pd.DataFrame({'rel_time': [-1], 'coef': [0], 'se': [0]})
    event_study_results = pd.concat([event_study_results, base_row]).sort_values('rel_time')
    
    return event_study_results

In [ ]:
# Test your function
def test_exercise_1():
    """Test event study implementation."""
    es_results = event_study_twfe(
        df_did, 'state_id', 'year', 'employment_rate', 'treated', 2012
    )
    
    assert es_results is not None, "Function returns None - did you implement it?"
    assert 'rel_time' in es_results.columns, "Missing rel_time column"
    assert 'coef' in es_results.columns, "Missing coef column"
    assert 'se' in es_results.columns, "Missing se column"
    
    # Plot event study
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Coefficients with 95% CI
    ax.errorbar(es_results['rel_time'], es_results['coef'], 
                yerr=1.96*es_results['se'], 
                marker='o', markersize=8, linewidth=2, capsize=5)
    
    ax.axhline(y=0, color='red', linestyle='--', linewidth=1.5)
    ax.axvline(x=-0.5, color='gray', linestyle='--', linewidth=1.5, 
               label='Treatment begins')
    
    ax.set_xlabel('Years Since Treatment', fontsize=12)
    ax.set_ylabel('Treatment Effect', fontsize=12)
    ax.set_title('Event Study: Treatment Effect Over Time', fontsize=14)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("✅ Correct! Event study implemented.")
    print("\nInterpretation:")
    print("  - Pre-treatment coefficients should be near zero (parallel trends)")
    print("  - Post-treatment coefficients show dynamic treatment effects")
    print("\nEstimated coefficients:")
    print(es_results)
    
    return True

# Uncomment to test
# test_exercise_1()

### Exercise 2: Placebo Test

**Task:** Conduct a placebo test by estimating treatment effect on an **unaffected outcome**.

Generate a placebo outcome that:
- Has entity and time FE
- Is NOT affected by minimum wage

Estimate DiD on this placebo outcome. Treatment effect should be zero.

**Hints:**
- Use same entity/time effects as real data
- But beta = 0 for treatment
- Should fail to reject H0: treatment effect = 0

In [ ]:
# YOUR CODE HERE
def placebo_test(df, entity_col, time_col, treatment_col, post_col):
    """
    Create placebo outcome and test for treatment effect.
    
    Parameters
    ----------
    df : pd.DataFrame
    entity_col : str
    time_col : str
    treatment_col : str
    post_col : str
    
    Returns
    -------
    dict
        Coefficient, SE, t-stat for placebo outcome
    """
    # TODO: Generate placebo outcome and estimate
    pass

In [ ]:
# SOLUTION (hidden)
def placebo_test_solution(df, entity_col, time_col, treatment_col, post_col):
    """Create placebo outcome and test for treatment effect."""
    # Generate placebo outcome (entity + time FE + noise, NO treatment effect)
    df_placebo = df.copy()
    
    # Entity FE
    entity_fe = df_placebo.groupby(entity_col)[entity_col].transform(
        lambda x: np.random.randn() * 10 + 50
    )
    
    # Time FE
    time_fe = df_placebo.groupby(time_col)[time_col].transform(
        lambda x: np.random.randn() * 3
    )
    
    # Placebo outcome (NO treatment effect)
    np.random.seed(999)
    df_placebo['placebo_outcome'] = entity_fe + time_fe + np.random.randn(len(df)) * 5
    
    # Create treatment indicator
    df_placebo['treated_post'] = df_placebo[treatment_col] * df_placebo[post_col]
    
    # Estimate TWFE
    df_placebo_panel = df_placebo.set_index([entity_col, time_col])
    
    model = PanelOLS(
        dependent=df_placebo_panel['placebo_outcome'],
        exog=df_placebo_panel[['treated_post']],
        entity_effects=True,
        time_effects=True
    )
    results = model.fit(cov_type='clustered', cluster_entity=True)
    
    return {
        'coef': results.params['treated_post'],
        'se': results.std_errors['treated_post'],
        't_stat': results.tstats['treated_post'],
        'p_value': results.pvalues['treated_post']
    }

In [ ]:
# Test your function
def test_exercise_2():
    """Test placebo implementation."""
    placebo_result = placebo_test(df_did, 'state_id', 'year', 'treated', 'post')
    
    assert placebo_result is not None, "Function returns None - did you implement it?"
    assert 'coef' in placebo_result, "Missing coefficient"
    assert 't_stat' in placebo_result, "Missing t-statistic"
    
    print("✅ Correct! Placebo test implemented.")
    print("\nPlacebo Test Results:")
    print("=" * 50)
    print(f"Coefficient: {placebo_result['coef']:.4f}")
    print(f"Std Error: {placebo_result['se']:.4f}")
    print(f"t-statistic: {placebo_result['t_stat']:.3f}")
    print(f"p-value: {placebo_result['p_value']:.4f}")
    
    if abs(placebo_result['t_stat']) < 1.96:
        print("\n✅ PASS: Cannot reject H0 (no effect on placebo outcome)")
        print("   This is expected - placebo outcome shouldn't be affected.")
    else:
        print("\n⚠️  WARNING: Significant effect on placebo outcome detected")
        print("   This suggests specification issues in main analysis.")
    
    return True

# Uncomment to test
# test_exercise_2()

---

## Summary

### Key Takeaways

1. **TWFE Controls Two Sources of Confounding:**
   - Entity FE: Time-invariant entity characteristics
   - Time FE: Common shocks affecting all entities

2. **When to Use TWFE:**
   - Aggregate trends/shocks present
   - Business cycles, policy changes, seasonality
   - Always test with F-test

3. **DiD as Special Case:**
   - Treatment/control groups + pre/post periods
   - TWFE with treatment indicator
   - Requires parallel trends assumption

4. **Event Studies:**
   - Dynamic treatment effects
   - Test pre-trends (parallel trends)
   - Flexible treatment timing

5. **Limitations:**
   - Heterogeneous treatment effects (warning!)
   - Staggered treatment timing issues
   - See recent econometrics literature (Goodman-Bacon, Sun & Abraham)

### Best Practices

- Always use clustered standard errors
- Test for time FE significance
- Check parallel trends visually
- Conduct event study/robustness checks
- Report specifications with/without time FE

### What's Next

In Module 3, we'll study **random effects** models, which make different assumptions and can be more efficient when entity effects are uncorrelated with predictors.

---

**Next:** Module 3 - Random Effects Models

In [ ]:
key_takeaways(["Next:"])